## 2) Reasoning y Function Calling
Objetivo: Practicar con modelos razonadores y ver el function calling

### 2.1- Razonamiento
Desplegar un modelo razonador y parametrizar distintos grados de razonamiento (low, medium, high)

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning?tabs=csharp%2Cgpt-5)


### Configuración local de variables de entorno

Antes de ejecutar las celdas de esta parte, crea un archivo `.env` en la raíz del proyecto copiando `.env.example` y completando los valores reales. El notebook carga esas variables automáticamente al inicio para reutilizar el mismo entorno sin escribir endpoints ni credenciales en el código.


In [3]:
import os
from pathlib import Path

env_path = Path("../.env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value
    print("Variables cargadas desde .env")
else:
    print("No se encontró .env; usando variables ya definidas en el sistema")

Variables cargadas desde .env


In [4]:
import os
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import time

base_url = os.getenv("AZURE_FOUNDRY_OPENAI_BASE_URL")
if not base_url:
    raise ValueError("Falta AZURE_FOUNDRY_OPENAI_BASE_URL en el entorno (.env).")

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
 )

client = OpenAI(
    base_url=base_url,
    api_key=token_provider,
 )

reasoning_model = os.getenv("AZURE_OPENAI_REASONING_DEPLOYMENT", "gpt-5-mini")
prompt = (
    "Define una politica de uso de IA para una universidad: privacidad, evaluacion, "
    "plagio, transparencia y excepciones. Incluye conflictos entre principios y como resolverlos."
 )

for effort in ["low", "medium", "high"]:
    print(f"\n===== REASONING {effort.upper()} =====")
    inicio = time.perf_counter()

    response = client.responses.create(
        input=prompt,
        model=reasoning_model,
        reasoning={
            "effort": effort,
            "summary": "auto"
        },
        text={
            "verbosity": "low"
        }
    )

    duracion_ms = round((time.perf_counter() - inicio) * 1000)
    print(f"Tiempo aprox: {duracion_ms} ms")
    print(f"Tokens salida: {response.usage.output_tokens}")
    print(response.output_text)


===== REASONING LOW =====
Tiempo aprox: 30388 ms
Tokens salida: 1375
Política de Uso de Inteligencia Artificial (IA) — Universidad

Objeto
- Establecer normas para el uso seguro, ético y responsable de herramientas y sistemas de IA por parte de estudiantes, docentes, personal administrativo e investigadores, garantizando privacidad, integridad académica, transparencia y excepciones justificadas.

Ámbito de aplicación
- Aplica a toda actividad académica, administrativa y de investigación en la universidad que implique sistemas o herramientas de IA (internas o de terceros).

1. Privacidad y protección de datos
- Principio: minimizar la recolección y el procesamiento de datos personales; garantizar confidencialidad y cumplimiento de la normativa de protección de datos aplicable.
- Reglas:
  - Antes de integrar una herramienta de IA, realizar evaluación de impacto de privacidad (EIP) si procesa datos personales o sensibles.
  - Prohibido subir datos sensibles (salud, etnia, información fi

### 2.2- Function calling
Activar un motor de búsqueda web para probar llamadas a funciones (`function calling`) que recuperen información externa.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/web-search)

⭐ Suma puntos usar deep research o hacer function calling con una función custom.

In [5]:
import os
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

base_url = os.getenv("AZURE_FOUNDRY_OPENAI_BASE_URL")
if not base_url:
    raise ValueError("Falta AZURE_FOUNDRY_OPENAI_BASE_URL en el entorno (.env).")

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
 )

client = OpenAI(
  base_url=base_url,
  api_key=token_provider,
 )

reasoning_model = os.getenv("AZURE_OPENAI_REASONING_DEPLOYMENT", "gpt-5-mini")
response = client.responses.create(
  model=reasoning_model,
  tools=[{"type": "web_search"}],
  input="Por favor busca las ultimas noticias del mundo del futbol español"
 )

print(response.output_text)

Perfecto — he buscado las últimas noticias del fútbol español y aquí tienes un resumen breve y actualizado hasta hoy, 12 de abril de 2026. Dime si quieres que amplíe cualquiera de los puntos (partido por partido, plantilla lesionados, clasificación, o enlaces a las crónicas completas).

- Nota institucional de LaLiga: LaLiga ha publicado varias notas informativas recientes (entre ellas referencia a la Sentencia 63/2026 relativa a los paros protagonizados por futbolistas y comunicaciones sobre medidas cautelares vinculadas al Getafe). ([laliga.com](https://www.laliga.com/noticias))

- Resultados y jornada en LaLiga: resumen de última jornada — por ejemplo, Getafe ganó al Athletic; Celta venció al Valencia; Real Oviedo derrotó al Sevilla; Alavés empató con Osasuna (resumen de la actualidad del día). ([es.besoccer.com](https://es.besoccer.com/noticia/actualidad-del-dia-en-el-futbol-espanol-a-5-de-abril-de-2026-1402894))

- FC Barcelona: continúa la polémica y las reacciones tras el Barça–

### Entregable: 
Notebook que muestre:
	- Ejemplos comparativos (misma tarea con distintos niveles de razonamiento).
	- Integración del web search y ejemplo de `function calling` que combine resultados externos con la respuesta del modelo.

---